In [131]:
import pandas as pd
import numpy as np
df_grab=pd.read_csv('ncr_ride_bookings.csv')
df_grab.head(5)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [132]:
df_grab.shape

(150000, 21)

In [133]:
df_grab.columns

Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='object')

In [134]:
df_grab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

In [135]:
df_grab.isnull().sum()

Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10500
Avg CTAT                              48000
Cancelled Rides by Customer          139500
Reason for cancelling by Customer    139500
Cancelled Rides by Driver            123000
Driver Cancellation Reason           123000
Incomplete Rides                     141000
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                        48000
dtype: int64

In [136]:
df_grab.duplicated().sum()

np.int64(0)

Oveview:
- This dataset contains 150,000 records and 21 columns.
- Initial investigation reveals that 8 columns do not contain null values ​​(Date, Time, BookingID, BookingStatus, Customer ID, Vehicle Type, Pickup Location, Drop Location). The remaining columns contain null values; however, this may be due to business logic. For example, the reason for cancellation is only displayed when the trip is canceled by the customer or driver. Further investigation is needed on this issue.
- No two rows have the same value.

In [137]:
df_grab.groupby('Booking Status')['Avg VTAT'].apply(lambda x: x.isnull().value_counts())
df_grab.groupby('Booking Status')['Avg CTAT'].apply(lambda x: x.isnull().value_counts())
df_grab.groupby('Booking Status')['Booking Value'].apply(lambda x: x.isnull().value_counts())
df_grab.groupby('Booking Status')['Ride Distance'].apply(lambda x: x.isnull().value_counts())
df_grab.groupby('Booking Status')['Payment Method'].apply(lambda x: x.isnull().value_counts())

Booking Status              
Cancelled by Customer  True     10500
Cancelled by Driver    True     27000
Completed              False    93000
Incomplete             False     9000
No Driver Found        True     10500
Name: Payment Method, dtype: int64

- After checking, I found no problem with the important columns having null values ​​based on the Booking Status groups; it's just a logical business practice.

In [138]:
df_grab.duplicated(subset=['Booking ID']).sum()

np.int64(1233)

After checking for duplicate data in the Booking ID column, I discovered 1233 duplicate entries. This data needs to be processed.

In [139]:
df_grab['Date']=pd.to_datetime(df_grab['Date'],errors='coerce')
df_grab['Time']=pd.to_datetime(df_grab['Time'],format='%H:%M:%S',errors='coerce').dt.time
df_grab['Datetime']=pd.to_datetime(df_grab['Date'].astype(str)+' '+df_grab['Time'].astype(str))
df_grab['Year']=pd.to_datetime(df_grab['Date']).dt.year
df_grab['Month']=pd.to_datetime(df_grab['Date']).dt.month
df_grab['Day']=pd.to_datetime(df_grab['Date']).dt.day
df_grab['Weekday']=pd.to_datetime(df_grab['Date']).dt.day_of_week
df_grab['Weekend']=np.where(df_grab['Weekday']>=5,1,0)
df_grab.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Datetime,Year,Month,Day,Weekday,Weekend
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,2024,3,23,5,1
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,2024,11,29,4,0
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,2024,8,23,4,0
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,2024,10,21,0,0
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,2024,9,16,0,0


In [140]:
df_grab=df_grab.sort_values(by=['Datetime'],ascending=True)
df_grab=df_grab.drop_duplicates(subset='Booking ID',keep='last')
df_grab.shape

(148767, 27)

- I created a data table in ascending order based on the Datetime column, and then handled duplicates by using the Booking ID of the most recent booking if two bookings were the same.
- After processing, only 148,767 records remain.

In [ ]:
from sqlalchemy import create_engine
username='hide'
password='hide'
host='hide'
port='hide'
database = 'hide'
engine=create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
tabel_name='hide'
df_grab.to_sql(tabel_name,engine,if_exists='replace',index=False)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18380\2022225748.py:9: UserWarning: The provided table name 'Booking' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_grab.to_sql(tabel_name,engine,if_exists='replace',index=False)


148767